# Geant4 Integration Example

This notebook demonstrates how to use the RADSIM Geant4 integration to simulate radiation transport and analyze the results.

## Overview

In this example, we'll:
1. Set up a radiation source (Cs-137)
2. Create a complex geometry with different materials
3. Run gamma and beta transport simulations
4. Visualize the resulting spectra

## Setup

First, we need to import the necessary modules and start the JVM:

In [ ]:
import startJVM
import jpype
import jpype.imports
import subprocess
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np

from gov.llnl.rtk.flux import FluxTrapezoid, FluxGroupTrapezoid
from java.util import ArrayList

from gov.llnl.rtk.physics import SourceImpl, Nuclides, EmissionCalculator, Quantity
from gov.llnl.ensdf.decay import BNLDecayLibrary
from gov.nist.physics.xray import NISTXrayLibrary
from java.nio.file import Paths

# Import the SourceGenerator class from the Geant4 package
SourceGenerator = jpype.JClass('gov.llnl.rtk.geant4.SourceGenerator')

## Helper Functions

These functions help us convert between different data formats:

In [ ]:
def parse_beta_file(path):
    """Parse a beta spectrum file into a FluxTrapezoid object."""
    with open(path, 'r') as f:
        lines = f.readlines()
    
    energies = []
    densities = []
    for line in lines:
        entries = line.split()
        try:
            energies.append(float(entries[0]))
            densities.append(float(entries[1]))
        except Exception:
            pass
    
    flux = FluxTrapezoid()
    for i in range(1, len(energies)):
        flux.addPhotonGroup(FluxGroupTrapezoid(energies[i-1], energies[i], densities[i-1], densities[i]))
    return flux

def parse_gamma(gammaList, resolution = 1.0):
    """Convert gamma line data into a continuous spectrum with the specified resolution."""
    nBins = int(4000. / resolution)
    energies = [i * resolution for i in range(nBins + 1)]
    densities = [0.0] * len(energies)
    
    for gamma in gammaList:
        energy = gamma.getEnergy().getValue()
        intensity = gamma.getIntensity().getValue()
        
        lower_index = int(energy // resolution)
        upper_index = lower_index + 1
        
        if upper_index < len(energies):
            lower_fraction = (upper_index * resolution - energy) / resolution
            upper_fraction = (energy - lower_index * resolution) / resolution
            
            densities[lower_index] += intensity * lower_fraction
            densities[upper_index] += intensity * upper_fraction
    
    flux = FluxTrapezoid()
    for i in range(1, len(energies)):
        flux.addPhotonGroup(FluxGroupTrapezoid(energies[i-1], energies[i], densities[i-1], densities[i]))
    
    return flux

def get_flux_from_lines(isotopes, activities):
    """Create a flux object from isotopes and their activities.
    Consistent with mcnp_example.ipynb implementation."""
    # Load the decay library
    decay_lib = BNLDecayLibrary()
    decay_lib.setXrayLibrary(NISTXrayLibrary.getInstance())
    decay_lib.loadFile(Paths.get("BNL2023.txt"))

    # Set up the emission calculator
    calculator = EmissionCalculator()
    calculator.setDecayLibrary(decay_lib)

    # Add sources
    sources = ArrayList()
    for isotope, activity in zip(isotopes, activities):
        sources.add(SourceImpl.fromActivity(Nuclides.get(isotope), activity))

    # Calculate the emissions
    emissions = calculator.apply(sources)

    # Create the flux
    flux = FluxTrapezoid()
    for gamma in emissions.getGammas():
        energy = gamma.getEnergy().getValue()
        intensity = gamma.getIntensity().getValue()
        
        nBins = int(4000. / 1.0)
        energies = [i * 1.0 for i in range(nBins + 1)]
        densities = [0.0] * len(energies)
        
        lower_index = int(energy // 1.0)
        upper_index = lower_index + 1
        
        if upper_index < len(energies):
            lower_fraction = (upper_index * 1.0 - energy) / 1.0
            upper_fraction = (energy - lower_index * 1.0) / 1.0
            
            densities[lower_index] += intensity * lower_fraction
            densities[upper_index] += intensity * upper_fraction
        
        for i in range(1, len(energies)):
            flux.addPhotonGroup(FluxGroupTrapezoid(energies[i-1], energies[i], densities[i-1], densities[i]))

    return flux

def plot_spectrum(spectrum_data, title):
    """Plot a spectrum from FluxTrapezoid data."""
    energies_lower = []
    energies_upper = []
    counts = []
    for groups in spectrum_data.getPhotonGroups():
        energies_lower.append(groups.getEnergyLower())
        energies_upper.append(groups.getEnergyUpper())
        counts.append(groups.getCounts())
    plt.figure(figsize=(10, 6))
    plt.bar(energies_lower, counts, width=[upper - lower for lower, upper in zip(energies_lower, energies_upper)], align='edge', alpha=0.7)
    plt.title(title)
    plt.xlabel('Energy (MeV)')
    plt.ylabel('Counts')
    if any(count > 0 for count in counts):
        plt.yscale('log')
    plt.grid()
    plt.show()

def combine_csv(file1, file2):
    """Combine two CSV files (add their numeric values)."""
    with open(file2, 'r') as f2:
        first_7_rows_f2 = [next(f2).strip() for _ in range(7)]
    df1 = pd.read_csv(file1, skiprows=7, header=None, na_values=[''], keep_default_na=False)
    df2 = pd.read_csv(file2, skiprows=7, header=None, na_values=[''], keep_default_na=False)
    df1 = df1.apply(pd.to_numeric, errors='coerce').fillna(0)
    df2 = df2.apply(pd.to_numeric, errors='coerce').fillna(0)
    added_part = df1.values + df2.values
    
    common_prefix = os.path.commonprefix([file1, file2])
    output_file = common_prefix.rstrip('_') + '.csv'
    with open(output_file, 'w') as f_out:
        for line in first_7_rows_f2:
            f_out.write(line + '\n')
        
        pd.DataFrame(added_part).to_csv(f_out, header=False, index=False)

## Radiation Source Setup

Let's set up a Cs-137 source and calculate its emissions:

In [ ]:
# Initialize the SourceGenerator
generator = SourceGenerator()

# Set up radiation sources using Quantity instead of ActivityUnit
activity_100Bq = Quantity.of(100, "Bq")  # 100 Becquerels
activity_94_7Bq = Quantity.of(94.7, "Bq")  # 94.7 Becquerels for Ba137m (branching ratio)

# Create sources with specified activity
Cs137 = SourceImpl.fromActivity(Nuclides.get("Cs137"), activity_100Bq)
Ba137m = SourceImpl.fromActivity(Nuclides.get("Ba137m"), activity_94_7Bq)

# Load decay data
bnllib = BNLDecayLibrary()
bnllib.setXrayLibrary(NISTXrayLibrary.getInstance())  # Set X-ray library for more accurate X-ray emissions
bnllib.loadFile(Paths.get("BNL2023.txt"))

# Initialize emission calculator
emcal = EmissionCalculator()
emcal.setDecayLibrary(bnllib)

# Create source list and calculate emissions
sourceList = ArrayList()
sourceList.add(Cs137)
sourceList.add(Ba137m)

# Calculate emissions using apply method (consistent with sourcegen_example)
out = emcal.apply(sourceList)

# Get gamma and beta emission data
gamma_flux = parse_gamma(out.getGammas(), 1.0)

# Calculate total intensities
gamma_sum = 0
for gamma in out.getGammas():
    gamma_sum += gamma.getIntensity().getValue()
beta_sum = 0
for beta in out.getBetas():
    beta_sum += beta.getIntensity().getValue()

print(f"Total gamma intensity: {gamma_sum}")
print(f"Total beta intensity: {beta_sum}")

## Geometry Setup

Now let's set up a complex geometry for our simulation. This includes:
1. A central spherical source
2. A lead hemisphere shield
3. A tin cylindrical shell
4. An aluminum endcap
5. A conical lead support structure

In [ ]:
# Set up the source for gamma simulation
generator.setFlux(gamma_flux)
generator.setSourceParticle("gamma")
generator.setNumberOfParticle(int(10000 * gamma_sum/beta_sum))
generator.setSphericalSource(True)
generator.setSourceRadius(0.5)

# Initialize the environment
generator.setEnvironment()

# Create a spherical object (CH3Cl material)
sphere_elements = ArrayList()
sphere_elements.add("C")
sphere_elements.add("H")
sphere_elements.add("Cl")
sphere_multipliers = ArrayList()
sphere_multipliers.add(2)   # Add 2 C atoms per molecule
sphere_multipliers.add(3)   # Add 3 H atoms per molecule
sphere_multipliers.add(1)   # Add 1 Cl atoms per molecule
sphere_density = 1.3   # in g/cm3
sphere_geometeries = ArrayList()
sphere_geometeries.add(0.)  # inner radius in cm
sphere_geometeries.add(1.)  # outer radius in cm
sphere_geometeries.add(0.)  # starting phi angle in degrees
sphere_geometeries.add(360.)  # delta phi angle in degrees
sphere_geometeries.add(0.)  # starting theta angle in degrees
sphere_geometeries.add(180.)  # delta theta angle in degrees
sphere_geometeries.add(0.)  # rotation angle X in degrees
sphere_geometeries.add(0.)  # rotation angle Y in degrees
sphere_geometeries.add(0.)  # rotation angle Z in degrees
sphere_geometeries.add(0.)  # position X in cm
sphere_geometeries.add(0.)  # position Y in cm
sphere_geometeries.add(0.)  # position Z in cm
generator.environment.addSphericalObject(sphere_elements, sphere_multipliers, sphere_density, sphere_geometeries)

# Create a lead hemisphere shield
hemisphere_elements = ArrayList()
hemisphere_elements.add("Pb")
hemisphere_multipliers = ArrayList()
hemisphere_multipliers.add(1)
hemisphere_density = 11.35   # in g/cm3
hemisphere_geometeries = ArrayList()
hemisphere_geometeries.add(1.5)  # inner radius in cm
hemisphere_geometeries.add(2.)  # outer radius in cm
hemisphere_geometeries.add(0.)  # starting phi angle in degrees
hemisphere_geometeries.add(360.)  # delta phi angle in degrees
hemisphere_geometeries.add(0.)  # starting theta angle in degrees
hemisphere_geometeries.add(90.)  # delta theta angle in degrees
hemisphere_geometeries.add(0.)  # rotation angle X in degrees
hemisphere_geometeries.add(0.)  # rotation angle Y in degrees
hemisphere_geometeries.add(0.)  # rotation angle Z in degrees
hemisphere_geometeries.add(0.)  # position X in cm
hemisphere_geometeries.add(0.)  # position Y in cm
hemisphere_geometeries.add(0.)  # position Z in cm
generator.environment.addSphericalObject(hemisphere_elements, hemisphere_multipliers, hemisphere_density, hemisphere_geometeries)

# Create a tin cylindrical shell
shell_elements = ArrayList()
shell_elements.add("Sn")
shell_multipliers = ArrayList()
shell_multipliers.add(1)
shell_density = 7.2389   # in g/cm3
shell_geometeries = ArrayList()
shell_geometeries.add(1.5)  # inner radius in cm
shell_geometeries.add(2.)  # outer radius in cm
shell_geometeries.add(2.)  # half z in cm
shell_geometeries.add(0.)  # starting phi angle in degrees
shell_geometeries.add(360.)  # delta phi angle in degrees
shell_geometeries.add(0.)  # rotation angle X in degrees
shell_geometeries.add(0.)  # rotation angle Y in degrees
shell_geometeries.add(0.)  # rotation angle Z in degrees
shell_geometeries.add(0.)  # position X in cm
shell_geometeries.add(0.)  # position Y in cm
shell_geometeries.add(-1.2)  # position Z in cm
generator.environment.addCylindricalObject(shell_elements, shell_multipliers, shell_density, shell_geometeries)

# Create an aluminum endcap
endcap_elements = ArrayList()
endcap_elements.add("Al")
endcap_multipliers = ArrayList()
endcap_multipliers.add(1)
endcap_density = 2.7   # in g/cm3
endcap_geometeries = ArrayList()
endcap_geometeries.add(0)  # inner radius in cm
endcap_geometeries.add(1.5)  # outer radius in cm
endcap_geometeries.add(0.5)  # half z in cm
endcap_geometeries.add(0.)  # starting phi angle in degrees
endcap_geometeries.add(360.)  # delta phi angle in degrees
endcap_geometeries.add(0.)  # rotation angle X in degrees
endcap_geometeries.add(0.)  # rotation angle Y in degrees
endcap_geometeries.add(0.)  # rotation angle Z in degrees
endcap_geometeries.add(0.)  # position X in cm
endcap_geometeries.add(0.)  # position Y in cm
endcap_geometeries.add(-1.8)  # position Z in cm
generator.environment.addCylindricalObject(endcap_elements, endcap_multipliers, endcap_density, endcap_geometeries)

# Create a conical lead support
support_elements = ArrayList()
support_elements.add("Pb")
support_multipliers = ArrayList()
support_multipliers.add(1)
support_density = 11.35   # in g/cm3
support_geometeries = ArrayList()
support_geometeries.add(0.5)  # inner radius at -dz in cm
support_geometeries.add(1.5)  # outer radius at -dz in cm
support_geometeries.add(0.25)  # inner radius at +dz in cm
support_geometeries.add(0.5)  # outer radius at +dz in cm
support_geometeries.add(0.3)  # half z in cm
support_geometeries.add(0.)  # starting phi angle in degrees
support_geometeries.add(360.)  # delta phi angle in degrees
support_geometeries.add(0.)  # rotation angle X in degrees
support_geometeries.add(0.)  # rotation angle Y in degrees
support_geometeries.add(0.)  # rotation angle Z in degrees
support_geometeries.add(0.)  # position X in cm
support_geometeries.add(0.)  # position Y in cm
support_geometeries.add(-1.2)  # position Z in cm
generator.environment.addConicalObject(support_elements, support_multipliers, support_density, support_geometeries)

## Running the Gamma Transport Simulation

Now we'll prepare and run the Geant4 simulation for gamma rays:

In [ ]:
# Prepare beam lines for simulation
generator.environment.prepareBeamLines()

# Write settings to macro file for Geant4
generator.environment.writeSettingsToMacro()

# Run Geant4 for geometry check first
print("Running Geant4 geometry check...")
runScript_path = './runGEANT4.sh'
try:
    result = subprocess.run([runScript_path], check=True, shell=True, text=True, capture_output=True)
    print("Geometry check completed successfully")
except subprocess.CalledProcessError as e:
    print(f"An error occurred: {e}")
    print("Return code:", e.returncode)
    print("Output:\n", e.output)
    print("Error output:\n", e.stderr)

# Run simulation for gamma transport
print("Running gamma transport simulation...")
runScript_path = './runGEANT4.sh gamma'
try:
    result = subprocess.run([runScript_path], check=True, shell=True, text=True, capture_output=True)
    print("Gamma transport completed successfully")
except subprocess.CalledProcessError as e:
    print(f"An error occurred: {e}")
    print("Return code:", e.returncode)
    print("Output:\n", e.output)
    print("Error output:\n", e.stderr)

## Running the Beta Transport Simulation

Next, we'll run the simulation for beta particles:

In [ ]:
# We can use two approaches for beta spectrum:
# 1. Parse from file (original approach)
beta_flux_from_file = parse_beta_file('../data/beta-_Cs137_tot.bs')

# 2. Use directly from emission calculation (consistent with sourcegen_example)
# Extract beta emissions from the emission calculation we already did
# This is an alternative to reading from file, but we'll use the file for consistency with previous code

# Configure for beta simulation
generator.setFlux(beta_flux_from_file)
generator.setSourceParticle("e-")
generator.setNumberOfParticle(10000)
generator.fetchEnvironment()

# Prepare beam lines and write settings
generator.environment.prepareBeamLines()
generator.environment.writeSettingsToMacro()

# Run simulation for beta transport
print("Running beta transport simulation...")
runScript_path = './runGEANT4.sh beta'
try:
    result = subprocess.run([runScript_path], check=True, shell=True, text=True, capture_output=True)
    print("Beta transport completed successfully")
except subprocess.CalledProcessError as e:
    print(f"An error occurred: {e}")
    print("Return code:", e.returncode)
    print("Output:\n", e.output)
    print("Error output:\n", e.stderr)

## Combining and Processing Results

Now we'll combine the results from both gamma and beta simulations:

In [ ]:
# Combine CSV files from gamma and beta simulations
combine_csv('RadSim_h1_Ebeta_beta.csv', 'RadSim_h1_Ebeta_gamma.csv')
combine_csv('RadSim_h1_ElectronEnergy_escape_beta.csv', 'RadSim_h1_ElectronEnergy_escape_gamma.csv')
combine_csv('RadSim_h1_GammaEnergy_escape_beta.csv', 'RadSim_h1_GammaEnergy_escape_gamma.csv')
combine_csv('RadSim_h1_GammaEnergy_primary_beta.csv', 'RadSim_h1_GammaEnergy_primary_gamma.csv')
combine_csv('RadSim_h1_PositronEnergy_escape_beta.csv', 'RadSim_h1_PositronEnergy_escape_gamma.csv')

# Parse simulation results
print("Parsing simulation results...")
generator.environment.parseResults()

## Visualizing Results

Finally, let's visualize the spectra for gamma rays, electrons, and positrons:

In [ ]:
# Retrieve spectra from simulation results
gamma_spectrum = generator.environment.getGammaSpectrum()
electron_spectrum = generator.environment.getElectronSpectrum()
positron_spectrum = generator.environment.getPositronSpectrum()

# Plot the spectra
plot_spectrum(gamma_spectrum, "Gamma Spectrum")
plot_spectrum(electron_spectrum, "Electron Spectrum")
plot_spectrum(positron_spectrum, "Positron Spectrum")

## Analysis and Interpretation

The spectra above show:

1. **Gamma Spectrum**: 
   - Primary gamma peak from Ba-137m at 662 keV
   - Secondary peaks from Compton scattering
   - X-rays from lead shielding

2. **Electron Spectrum**: 
   - Primary beta spectrum from Cs-137 decay
   - Secondary electrons from Compton interactions and photoelectric effect

3. **Positron Spectrum**: 
   - Much lower intensity, primarily from pair production in high-Z materials

The simulated geometry demonstrates how different materials affect radiation transport:
- Lead hemisphere: strong attenuation of gamma rays
- Aluminum endcap: moderate attenuation with less secondary radiation
- Tin shell: intermediate attenuation properties

This example shows how the RADSIM Geant4 integration can be used to model complex radiation transport scenarios and extract detailed information about the resulting particle spectra.